# Biohub EDA and Local Validation

Bu notebook'un amaci veriyi ciddi sekilde tanimak: dosya butunlugu, Zarr/GEFF metadata, GT yogunluk, hareket mesafeleri, division olaylari, intensity dagilimlari ve baseline-vs-GT local validation.

Varsayilan ayarlar guvenlidir: tum movie'leri RAM'e almaz, once quick sample taramasi yapar. Full taramalar icin flag'leri bilincli olarak `True` yap.

## 1. Drive ve Paketler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q zarr numcodecs scipy scikit-image matplotlib tqdm pandas numpy

## 2. Proje Kodunu Yukle

In [ ]:
from pathlib import Path
import json
import math
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
if (PROJECT_DIR / 'biohub_baseline.py').exists():
    sys.path.insert(0, str(PROJECT_DIR))
elif (Path.cwd() / 'biohub_baseline.py').exists():
    sys.path.insert(0, str(Path.cwd()))
else:
    raise FileNotFoundError('biohub_baseline.py bulunamadi. Notebook proje klasorunden calismali.')

import biohub_baseline as bh

bh.find_dataset_base_dir(PROJECT_DIR / 'data')
print('BASE_DIR =', bh.BASE_DIR)
print('dataset_available =', bh.dataset_available(bh.BASE_DIR))

## 3. Analiz Ayarlari

In [ ]:
QUICK_MAX_SAMPLES = 5
RUN_FULL_INTEGRITY = False
RUN_FULL_GEFF_SUMMARY = False
RUN_INTENSITY_SCAN = True
RUN_BASELINE_LOCAL_VALIDATION = False

SCALES = np.array([1.625, 0.40625, 0.40625], dtype=float)
MATCH_MAX_DISTANCE_UM = 7.0

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

## 4. Dataset Overview

In [ ]:
overview = bh.inspect_dataset(bh.BASE_DIR)

train_dir = bh.BASE_DIR / 'train'
test_dir = bh.BASE_DIR / 'test'
train_zarr_paths = sorted(train_dir.glob('*.zarr'))
train_geff_paths = sorted(train_dir.glob('*.geff'))
test_zarr_paths = sorted(test_dir.glob('*.zarr'))

def embryo_id(sample_id):
    return str(sample_id).split('_')[0]

sample_rows = []
for split, paths in [('train_zarr', train_zarr_paths), ('train_geff', train_geff_paths), ('test_zarr', test_zarr_paths)]:
    for p in paths:
        sample_rows.append({'split': split, 'sample_id': p.stem, 'embryo_id': embryo_id(p.stem), 'path': str(p)})
samples_df = pd.DataFrame(sample_rows)
display(samples_df.groupby(['split', 'embryo_id']).size().rename('count').reset_index())
display(samples_df.head())

## 5. Integrity Check Fonksiyonlari

Bu bolum metadata ve secili timepoint okumalarini kontrol eder. Full volume RAM'e alinmaz.

In [ ]:
def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f)


def find_key_recursive(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for value in obj.values():
            found = find_key_recursive(value, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_key_recursive(value, key)
            if found is not None:
                return found
    return None


def expected_timepoints(shape):
    t = int(shape[0])
    return sorted(set([0, max(0, t // 2), max(0, t - 1)]))


def check_image_zarr(sample_path):
    sample_path = Path(sample_path)
    row = {
        'sample_id': sample_path.stem,
        'path': str(sample_path),
        'kind': 'image_zarr',
        'ok': True,
        'error': '',
        'shape': None,
        'dtype': None,
        'chunks': None,
        'read_timepoints': None,
    }
    try:
        if not (sample_path / '0' / 'zarr.json').exists():
            raise FileNotFoundError(f'Missing array metadata: {sample_path / "0" / "zarr.json"}')
        img = bh.open_image_zarr(sample_path, print_info=False)
        row['shape'] = tuple(int(x) for x in img.shape)
        row['dtype'] = str(img.dtype)
        row['chunks'] = tuple(int(x) for x in getattr(img, 'chunks', ()))
        if len(img.shape) != 4:
            raise ValueError(f'Expected 4D TZYX image, got shape={img.shape}')
        if str(img.dtype) != 'uint16':
            row['ok'] = False
            row['error'] += f'Unexpected dtype {img.dtype}; '
        read_ts = []
        for t in expected_timepoints(img.shape):
            _ = np.asarray(img[t, 0, 0:8, 0:8])
            read_ts.append(t)
        row['read_timepoints'] = read_ts
    except Exception as exc:
        row['ok'] = False
        row['error'] = repr(exc)
    return row


def check_geff(geff_path):
    geff_path = Path(geff_path)
    row = {
        'sample_id': geff_path.stem,
        'path': str(geff_path),
        'kind': 'geff',
        'ok': True,
        'error': '',
        'n_nodes': 0,
        'n_edges': 0,
        't_min': np.nan,
        't_max': np.nan,
        'division_nodes': 0,
        'estimated_number_of_nodes': np.nan,
    }
    try:
        required = [
            geff_path / 'nodes' / 'ids',
            geff_path / 'nodes' / 'props' / 't' / 'values',
            geff_path / 'nodes' / 'props' / 'z' / 'values',
            geff_path / 'nodes' / 'props' / 'y' / 'values',
            geff_path / 'nodes' / 'props' / 'x' / 'values',
            geff_path / 'edges' / 'ids',
        ]
        missing = [str(p) for p in required if not p.exists()]
        if missing:
            raise FileNotFoundError('Missing GEFF arrays: ' + '; '.join(missing[:3]))
        nodes_df, edges_df = bh.load_geff_annotations(geff_path)
        row['n_nodes'] = len(nodes_df)
        row['n_edges'] = len(edges_df)
        if len(nodes_df):
            row['t_min'] = int(nodes_df['t'].min())
            row['t_max'] = int(nodes_df['t'].max())
        if len(edges_df):
            out_counts = edges_df['source_id'].value_counts()
            row['division_nodes'] = int((out_counts >= 2).sum())
        root_meta = read_json_safe(geff_path / 'zarr.json')
        estimated = find_key_recursive(root_meta, 'estimated_number_of_nodes') if root_meta else None
        if estimated is not None:
            row['estimated_number_of_nodes'] = estimated
    except Exception as exc:
        row['ok'] = False
        row['error'] = repr(exc)
    return row


def validate_dataset_integrity(base_dir, max_samples=5):
    base_dir = Path(base_dir)
    rows = []
    train_z = sorted((base_dir / 'train').glob('*.zarr'))
    test_z = sorted((base_dir / 'test').glob('*.zarr'))
    train_g = sorted((base_dir / 'train').glob('*.geff'))
    if max_samples is not None:
        train_z = train_z[:max_samples]
        test_z = test_z[:max_samples]
        train_g = train_g[:max_samples]
    for p in tqdm(train_z, desc='check train zarr'):
        rows.append(check_image_zarr(p) | {'split': 'train'})
    for p in tqdm(test_z, desc='check test zarr'):
        rows.append(check_image_zarr(p) | {'split': 'test'})
    for p in tqdm(train_g, desc='check train geff'):
        rows.append(check_geff(p) | {'split': 'train'})
    return pd.DataFrame(rows)

## 6. Integrity Check Calistir

In [ ]:
max_samples = None if RUN_FULL_INTEGRITY else QUICK_MAX_SAMPLES
integrity_df = validate_dataset_integrity(bh.BASE_DIR, max_samples=max_samples)
display(integrity_df)
display(integrity_df.groupby(['kind', 'ok']).size().rename('count').reset_index())

bad = integrity_df[~integrity_df['ok']]
if len(bad):
    display(bad[['split', 'kind', 'sample_id', 'error']])
else:
    print('Integrity quick check passed.')

## 7. Zarr Metadata Ozeti

In [ ]:
def collect_zarr_metadata(paths, split):
    rows = []
    for p in tqdm(paths, desc=f'zarr metadata {split}'):
        try:
            img = bh.open_image_zarr(p, print_info=False)
            shape = tuple(int(x) for x in img.shape)
            rows.append({
                'split': split,
                'sample_id': p.stem,
                'embryo_id': embryo_id(p.stem),
                'T': shape[0],
                'Z': shape[1],
                'Y': shape[2],
                'X': shape[3],
                'dtype': str(img.dtype),
                'chunks': tuple(int(x) for x in getattr(img, 'chunks', ())),
                'n_voxels': int(np.prod(shape)),
                'approx_uint16_gib': np.prod(shape) * 2 / (1024 ** 3),
            })
        except Exception as exc:
            rows.append({'split': split, 'sample_id': p.stem, 'error': repr(exc)})
    return pd.DataFrame(rows)


zarr_meta_df = pd.concat([
    collect_zarr_metadata(train_zarr_paths, 'train'),
    collect_zarr_metadata(test_zarr_paths, 'test'),
], ignore_index=True)

display(zarr_meta_df.head())
display(zarr_meta_df.groupby('split')[['T', 'Z', 'Y', 'X', 'approx_uint16_gib']].agg(['min', 'median', 'max']))
display(zarr_meta_df.groupby(['split', 'dtype']).size().rename('count').reset_index())
display(zarr_meta_df.groupby(['split', 'chunks']).size().rename('count').reset_index())

## 8. GEFF / GT Ozeti ve Division Analizi

In [ ]:
def geff_sample_summary(geff_path):
    nodes_df, edges_df = bh.load_geff_annotations(geff_path)
    out_counts = edges_df['source_id'].value_counts() if len(edges_df) else pd.Series(dtype=int)
    division_nodes = out_counts[out_counts >= 2]
    root_meta = read_json_safe(Path(geff_path) / 'zarr.json')
    estimated = find_key_recursive(root_meta, 'estimated_number_of_nodes') if root_meta else None
    return {
        'sample_id': Path(geff_path).stem,
        'embryo_id': embryo_id(Path(geff_path).stem),
        'n_gt_nodes': len(nodes_df),
        'n_gt_edges': len(edges_df),
        't_min': int(nodes_df['t'].min()) if len(nodes_df) else np.nan,
        't_max': int(nodes_df['t'].max()) if len(nodes_df) else np.nan,
        'gt_nodes_per_t_median': float(nodes_df.groupby('t').size().median()) if len(nodes_df) else 0,
        'gt_nodes_per_t_max': int(nodes_df.groupby('t').size().max()) if len(nodes_df) else 0,
        'division_nodes': int(len(division_nodes)),
        'max_out_degree': int(out_counts.max()) if len(out_counts) else 0,
        'estimated_number_of_nodes': estimated,
    }


geff_paths_for_summary = train_geff_paths if RUN_FULL_GEFF_SUMMARY else train_geff_paths[:QUICK_MAX_SAMPLES]
geff_summary_df = pd.DataFrame([geff_sample_summary(p) for p in tqdm(geff_paths_for_summary, desc='GEFF summary')])
display(geff_summary_df)
if len(geff_summary_df):
    display(geff_summary_df.describe(include='all'))
    display(geff_summary_df.groupby('embryo_id')[['n_gt_nodes', 'n_gt_edges', 'division_nodes']].sum().reset_index())

## 9. GT Edge Mesafeleri ve Hareket Hizlari

In [ ]:
def gt_edge_distance_table(geff_path):
    nodes_df, edges_df = bh.load_geff_annotations(geff_path)
    if len(nodes_df) == 0 or len(edges_df) == 0:
        return pd.DataFrame()
    src = nodes_df.rename(columns={'node_id': 'source_id', 't': 't_src', 'z': 'z_src', 'y': 'y_src', 'x': 'x_src'})
    tgt = nodes_df.rename(columns={'node_id': 'target_id', 't': 't_tgt', 'z': 'z_tgt', 'y': 'y_tgt', 'x': 'x_tgt'})
    merged = edges_df.merge(src, on='source_id', how='left').merge(tgt, on='target_id', how='left')
    dz = (merged['z_tgt'] - merged['z_src']).to_numpy(dtype=float)
    dy = (merged['y_tgt'] - merged['y_src']).to_numpy(dtype=float)
    dx = (merged['x_tgt'] - merged['x_src']).to_numpy(dtype=float)
    merged['distance_um'] = np.sqrt((dz * SCALES[0]) ** 2 + (dy * SCALES[1]) ** 2 + (dx * SCALES[2]) ** 2)
    merged['dt'] = merged['t_tgt'] - merged['t_src']
    merged['sample_id'] = Path(geff_path).stem
    return merged[['sample_id', 'source_id', 'target_id', 't_src', 't_tgt', 'dt', 'distance_um']]


edge_distance_parts = [gt_edge_distance_table(p) for p in tqdm(geff_paths_for_summary, desc='GT edge distances')]
edge_distance_df = pd.concat([x for x in edge_distance_parts if len(x)], ignore_index=True) if edge_distance_parts else pd.DataFrame()
display(edge_distance_df.head())
if len(edge_distance_df):
    display(edge_distance_df[['dt', 'distance_um']].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
    ax = edge_distance_df['distance_um'].clip(upper=50).hist(bins=50, figsize=(8, 4))
    ax.set_title('GT edge distances, clipped at 50 um')
    ax.set_xlabel('distance_um')
    plt.show()

## 10. Intensity Sampling

Tum volume'u yuklemeden secili timepointlerde percentile dagilimlarini inceler.

In [ ]:
def intensity_summary_for_sample(sample_path, max_timepoints=5):
    img = bh.open_image_zarr(sample_path, print_info=False)
    ts = np.linspace(0, img.shape[0] - 1, num=min(max_timepoints, img.shape[0]), dtype=int)
    rows = []
    for t in ts:
        vol = np.asarray(img[int(t)])
        rows.append({
            'sample_id': Path(sample_path).stem,
            't': int(t),
            'min': int(vol.min()),
            'p1': float(np.percentile(vol, 1)),
            'p50': float(np.percentile(vol, 50)),
            'p99': float(np.percentile(vol, 99)),
            'p99_8': float(np.percentile(vol, 99.8)),
            'max': int(vol.max()),
            'mean': float(vol.mean()),
        })
    return pd.DataFrame(rows)


if RUN_INTENSITY_SCAN:
    intensity_paths = train_zarr_paths[:min(QUICK_MAX_SAMPLES, len(train_zarr_paths))]
    intensity_df = pd.concat([intensity_summary_for_sample(p) for p in tqdm(intensity_paths, desc='intensity')], ignore_index=True)
    display(intensity_df)
    display(intensity_df.groupby('sample_id')[['p1', 'p50', 'p99', 'p99_8', 'max', 'mean']].median())
else:
    print('RUN_INTENSITY_SCAN False')

## 11. Visual Sanity Check

In [ ]:
if train_zarr_paths:
    sample_path = train_zarr_paths[0]
    sample_id = sample_path.stem
    img = bh.open_image_zarr(sample_path)
    geff_path = train_dir / f'{sample_id}.geff'
    if geff_path.exists():
        nodes_df, edges_df = bh.load_geff_annotations(geff_path)
        t0 = int(nodes_df['t'].median()) if len(nodes_df) else 0
        z0 = int(nodes_df[nodes_df['t'] == t0]['z'].median()) if len(nodes_df[nodes_df['t'] == t0]) else int(img.shape[1] // 2)
        bh.visualize_gt_overlay(img, nodes_df, t=t0, z=z0, z_tolerance=2)
    else:
        bh.visualize_slice(img, t=0, z=int(img.shape[1] // 2))

## 12. Baseline vs GT Local Validation Iskeleti

Bu bolum parametre tuning icin kullanilacak. Varsayilan olarak kapali, cunku detection calistirir.

In [ ]:
def match_nodes_by_time(pred_df, gt_df, max_distance_um=MATCH_MAX_DISTANCE_UM):
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial.distance import cdist

    matches = []
    for t in sorted(set(pred_df['t']).intersection(set(gt_df['t']))):
        p = pred_df[pred_df['t'] == t].reset_index(drop=True)
        g = gt_df[gt_df['t'] == t].reset_index(drop=True)
        if len(p) == 0 or len(g) == 0:
            continue
        p_um = p[['z', 'y', 'x']].to_numpy(dtype=float) * SCALES
        g_um = g[['z', 'y', 'x']].to_numpy(dtype=float) * SCALES
        dist = cdist(p_um, g_um)
        rows, cols = linear_sum_assignment(dist)
        for r, c in zip(rows, cols):
            d = float(dist[r, c])
            if d <= max_distance_um:
                matches.append({
                    't': int(t),
                    'pred_node_id': p.loc[r, 'node_id'],
                    'gt_node_id': g.loc[c, 'node_id'],
                    'distance_um': d,
                })
    return pd.DataFrame(matches)


def edge_match_score(pred_edges_df, gt_edges_df, matches_df):
    if len(pred_edges_df) == 0 or len(gt_edges_df) == 0 or len(matches_df) == 0:
        return {'edge_tp': 0, 'edge_fp': len(pred_edges_df), 'edge_fn': len(gt_edges_df), 'edge_jaccard_like': 0.0}
    pred_to_gt = dict(zip(matches_df['pred_node_id'], matches_df['gt_node_id']))
    gt_edges = set(map(tuple, gt_edges_df[['source_id', 'target_id']].to_numpy()))
    mapped_pred_edges = []
    for row in pred_edges_df.itertuples(index=False):
        if row.source_id in pred_to_gt and row.target_id in pred_to_gt:
            mapped_pred_edges.append((pred_to_gt[row.source_id], pred_to_gt[row.target_id]))
    mapped_pred_edges = set(mapped_pred_edges)
    tp = len(mapped_pred_edges & gt_edges)
    fp = len(mapped_pred_edges - gt_edges)
    fn = len(gt_edges - mapped_pred_edges)
    denom = tp + fp + fn
    return {'edge_tp': tp, 'edge_fp': fp, 'edge_fn': fn, 'edge_jaccard_like': tp / denom if denom else 0.0}


def run_small_local_validation(sample_path, max_timepoints=5):
    sample_id = Path(sample_path).stem
    img = bh.open_image_zarr(sample_path, print_info=False)
    geff_path = Path(sample_path).with_suffix('.geff')
    nodes_df, gt_edges_df = bh.load_geff_annotations(geff_path)
    gt_nodes_limited = nodes_df[nodes_df['t'] < max_timepoints].copy()
    gt_edges_limited = gt_edges_df.merge(
        gt_nodes_limited[['node_id']].rename(columns={'node_id': 'source_id'}), on='source_id'
    ).merge(
        gt_nodes_limited[['node_id']].rename(columns={'node_id': 'target_id'}), on='target_id'
    )
    pred_nodes = bh.detect_cells_for_sample(img, sample_id, bh.DETECTION_PARAMS, max_timepoints=max_timepoints)
    pred_edges = bh.link_detections(pred_nodes, bh.LINKING_PARAMS)
    matches = match_nodes_by_time(pred_nodes, gt_nodes_limited)
    node_precision = len(matches) / len(pred_nodes) if len(pred_nodes) else 0.0
    node_recall_sparse = len(matches) / len(gt_nodes_limited) if len(gt_nodes_limited) else 0.0
    edge_scores = edge_match_score(pred_edges, gt_edges_limited, matches)
    return {
        'sample_id': sample_id,
        'pred_nodes': len(pred_nodes),
        'gt_nodes_sparse': len(gt_nodes_limited),
        'matched_nodes': len(matches),
        'node_precision_vs_sparse_gt': node_precision,
        'node_recall_vs_sparse_gt': node_recall_sparse,
        'pred_edges': len(pred_edges),
        'gt_edges_sparse': len(gt_edges_limited),
        **edge_scores,
    }, pred_nodes, pred_edges, matches


if RUN_BASELINE_LOCAL_VALIDATION:
    result, pred_nodes, pred_edges, matches = run_small_local_validation(train_zarr_paths[0], max_timepoints=5)
    display(pd.DataFrame([result]))
    display(matches.head())
else:
    print('RUN_BASELINE_LOCAL_VALIDATION False. Acmak icin flagi True yap.')

## 13. Sonraki Kararlar

Bu notebook'tan sonra bakilacak sinyaller:

- Integrity hatasi varsa once veri/zip/unzip duzeltilir.
- GT edge mesafeleri linking threshold ve motion model icin kullanilir.
- Division sayilari one-to-two linking stratejisini belirler.
- Intensity percentile dagilimlari detection parametrelerini belirler.
- Local validation ciktisi parametre sweep icin kullanilir.